# Exploratory Analysis — Texas Afghan Support Program

This notebook explores synthetic ZIP-code-level program data across Austin, Dallas, Fort Worth, Houston, and San Antonio.

The analysis focuses on four questions:

1. Where are households, participants, and rental assistance concentrated?
2. How did enrollment, rental assistance, and Financial Wellness Course attendance change from July through September?
3. Which ZIP codes experienced the largest month-to-month increases or declines?
4. How are program volume, assistance, and course attendance related?

**Scope:** descriptive exploratory analysis. These aggregate data support comparisons, distributions, trends, and associations, but not individual-level or causal conclusions.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent

DATA = ROOT / "data" / "asp_zip_metrics.csv"

df = pd.read_csv(DATA, dtype={"zip_code": str})

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

df.head()

## 1. Data overview and quality checks

In [ ]:
quality = pd.Series({
    "Rows": len(df),
    "Unique ZIP codes": df["zip_code"].nunique(),
    "Metro areas": df["city"].nunique(),
    "Missing city": int(df["city"].isna().sum()),
    "Missing total rental assistance": int(df["ra_total"].isna().sum()),
    "Missing FWC total": int(df["fwc_total"].isna().sum()),
})

quality

In [ ]:
monthly_ra_sum = df["ra_july"] + df["ra_august"] + df["ra_september"]
ra_reconciliation_error = (monthly_ra_sum - df["ra_total"]).abs()

pd.Series({
    "Maximum RA reconciliation error": ra_reconciliation_error.max(),
    "ZIP codes with reconciliation error > $0.01": int((ra_reconciliation_error > 0.01).sum()),
})

The rental-assistance total reconciles to the sum of July, August, and September at the cent level. That internal-consistency check supports using the monthly and total measures together in the analysis.

## 2. Descriptive statistics

In [ ]:
overall = pd.Series({
    "ZIP codes": len(df),
    "Households": df["total_households"].sum(),
    "Participants": df["total_participants"].sum(),
    "Total rental assistance": df["ra_total"].sum(),
    "Median ZIP rental assistance": df["ra_total"].median(),
    "Total unique FWC attendees": df["fwc_total"].sum(),
})

overall

In [ ]:
df[
    [
        "total_households",
        "total_participants",
        "ra_total",
        "fwc_total",
        "ra_per_household",
        "participants_per_household",
    ]
].describe().T

ZIP-level rental assistance is strongly right-skewed: a relatively small number of ZIP codes account for substantially larger totals than the typical ZIP. This makes medians, quantiles, and concentration measures more informative than averages alone.

## 3. Metro-area comparison

In [ ]:
city = (
    df.groupby("city", as_index=False)
      .agg(
          zip_codes=("zip_code", "nunique"),
          households=("total_households", "sum"),
          participants=("total_participants", "sum"),
          rental_assistance=("ra_total", "sum"),
          fwc_attendees=("fwc_total", "sum"),
      )
)

city["ra_per_household"] = city["rental_assistance"] / city["households"]
city["participants_per_household"] = city["participants"] / city["households"]

city.sort_values("rental_assistance", ascending=False)

In [ ]:
plot_city = city.sort_values("rental_assistance")

plt.figure(figsize=(8, 5))
plt.barh(plot_city["city"], plot_city["rental_assistance"])
plt.xlabel("Rental assistance ($)")
plt.ylabel("Metro area")
plt.title("Rental Assistance by Metro Area")
plt.tight_layout()
plt.show()

## 4. Distribution and concentration

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(df["ra_total"], bins=30)
plt.xlabel("Total rental assistance per ZIP ($)")
plt.ylabel("Number of ZIP codes")
plt.title("Distribution of ZIP-Level Rental Assistance")
plt.tight_layout()
plt.show()

In [ ]:
top10 = (
    df.nlargest(10, "ra_total")[
        ["zip_code", "city", "total_households", "total_participants", "ra_total"]
    ]
    .copy()
)

top10["share_of_total_ra"] = top10["ra_total"] / df["ra_total"].sum()
top10

In [ ]:
top10_share = top10["ra_total"].sum() / df["ra_total"].sum()
print(f"Top 10 ZIP codes account for {top10_share:.1%} of total rental assistance.")

This concentration measure complements the choropleth map: the map shows where high-volume ZIP codes are located, while the concentration statistic quantifies how much of the total assistance they represent.

## 5. Monthly program dynamics

### 5.1 Enrollment

In [ ]:
enrollment_monthly = pd.Series({
    "July": df["enrolled_july_cumulative"].sum(),
    "August": df["enrolled_august_cumulative"].sum(),
    "September": df["enrolled_september_cumulative"].sum(),
})

enrollment_mom = enrollment_monthly.pct_change() * 100

pd.DataFrame({
    "Cumulative households enrolled": enrollment_monthly,
    "MoM change (%)": enrollment_mom,
})

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(enrollment_monthly.index, enrollment_monthly.values, marker="o")
plt.ylabel("Cumulative households enrolled")
plt.title("Cumulative Enrollment by Month")
plt.tight_layout()
plt.show()

Because enrollment is cumulative, the August and September values should be interpreted as the total enrolled by that point in time rather than as standalone monthly counts.

### 5.2 Rental assistance

In [ ]:
ra_monthly = pd.Series({
    "July": df["ra_july"].sum(),
    "August": df["ra_august"].sum(),
    "September": df["ra_september"].sum(),
})

ra_mom = ra_monthly.pct_change() * 100

pd.DataFrame({
    "Rental assistance": ra_monthly,
    "MoM change (%)": ra_mom,
})

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(ra_monthly.index, ra_monthly.values, marker="o")
plt.ylabel("Rental assistance ($)")
plt.title("Monthly Rental Assistance")
plt.tight_layout()
plt.show()

### 5.3 Financial Wellness Course attendance

In [ ]:
fwc_monthly = pd.Series({
    "July": df["fwc_july"].sum(),
    "August": df["fwc_august"].sum(),
    "September": df["fwc_september"].sum(),
})

fwc_mom = fwc_monthly.pct_change() * 100

pd.DataFrame({
    "Monthly FWC attendance": fwc_monthly,
    "MoM change (%)": fwc_mom,
})

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(fwc_monthly.index, fwc_monthly.values, marker="o")
plt.ylabel("Monthly attendees")
plt.title("Financial Wellness Course Attendance by Month")
plt.tight_layout()
plt.show()

Monthly FWC attendance counts unique attendees within each month. `fwc_total` is a different measure: it represents unique attendees across the entire July–September period, so the monthly values should not be summed to reconstruct the total.

## 6. Metro-area monthly trends

In [ ]:
city_monthly = (
    df.groupby("city")
      .agg(
          enrollment_july=("enrolled_july_cumulative", "sum"),
          enrollment_august=("enrolled_august_cumulative", "sum"),
          enrollment_september=("enrolled_september_cumulative", "sum"),
          ra_july=("ra_july", "sum"),
          ra_august=("ra_august", "sum"),
          ra_september=("ra_september", "sum"),
          fwc_july=("fwc_july", "sum"),
          fwc_august=("fwc_august", "sum"),
          fwc_september=("fwc_september", "sum"),
      )
)

city_monthly

In [ ]:
ra_city_plot = city_monthly[["ra_july", "ra_august", "ra_september"]].T
ra_city_plot.index = ["July", "August", "September"]

plt.figure(figsize=(9, 5))
for city_name in ra_city_plot.columns:
    plt.plot(ra_city_plot.index, ra_city_plot[city_name], marker="o", label=city_name)

plt.ylabel("Rental assistance ($)")
plt.title("Monthly Rental Assistance by Metro Area")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
fwc_city_plot = city_monthly[["fwc_july", "fwc_august", "fwc_september"]].T
fwc_city_plot.index = ["July", "August", "September"]

plt.figure(figsize=(9, 5))
for city_name in fwc_city_plot.columns:
    plt.plot(fwc_city_plot.index, fwc_city_plot[city_name], marker="o", label=city_name)

plt.ylabel("Monthly attendees")
plt.title("Monthly FWC Attendance by Metro Area")
plt.legend()
plt.tight_layout()
plt.show()

## 7. ZIP-level growth and decline

In [ ]:
df["enrollment_aug_mom_pct"] = np.where(
    df["enrolled_july_cumulative"] > 0,
    (df["enrolled_august_cumulative"] - df["enrolled_july_cumulative"])
    / df["enrolled_july_cumulative"] * 100,
    np.nan,
)

df["enrollment_sep_mom_pct"] = np.where(
    df["enrolled_august_cumulative"] > 0,
    (df["enrolled_september_cumulative"] - df["enrolled_august_cumulative"])
    / df["enrolled_august_cumulative"] * 100,
    np.nan,
)

df["fwc_aug_mom_pct"] = np.where(
    df["fwc_july"] > 0,
    (df["fwc_august"] - df["fwc_july"]) / df["fwc_july"] * 100,
    np.nan,
)

df["fwc_sep_mom_pct"] = np.where(
    df["fwc_august"] > 0,
    (df["fwc_september"] - df["fwc_august"]) / df["fwc_august"] * 100,
    np.nan,
)

df[
    [
        "zip_code",
        "city",
        "enrollment_aug_mom_pct",
        "enrollment_sep_mom_pct",
        "ra_aug_vs_july_pct",
        "ra_sep_vs_aug_pct",
        "fwc_aug_mom_pct",
        "fwc_sep_mom_pct",
    ]
].head()

### Largest August-to-September RA increases

In [ ]:
df.nlargest(10, "ra_sep_vs_aug_pct")[
    ["zip_code", "city", "ra_august", "ra_september", "ra_sep_vs_aug_pct"]
]

### Largest August-to-September RA declines

In [ ]:
df.nsmallest(10, "ra_sep_vs_aug_pct")[
    ["zip_code", "city", "ra_august", "ra_september", "ra_sep_vs_aug_pct"]
]

### Largest August-to-September enrollment increases

In [ ]:
df.nlargest(10, "enrollment_sep_mom_pct")[
    [
        "zip_code",
        "city",
        "enrolled_august_cumulative",
        "enrolled_september_cumulative",
        "enrollment_sep_mom_pct",
    ]
]

Percentage growth can be large when the prior-month base is small. The rankings should therefore be read alongside the underlying counts rather than interpreted from percentages alone.

## 8. Relationships among program measures

In [ ]:
corr = df[
    [
        "total_households",
        "total_participants",
        "ra_total",
        "fwc_total",
        "ra_per_household",
    ]
].corr()

corr

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(df["total_households"], df["ra_total"], alpha=0.65)
plt.xlabel("Households")
plt.ylabel("Rental assistance ($)")
plt.title("Households and Rental Assistance by ZIP Code")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(df["total_participants"], df["fwc_total"], alpha=0.65)
plt.xlabel("Participants")
plt.ylabel("Unique FWC attendees")
plt.title("Program Participation and FWC Attendance by ZIP Code")
plt.tight_layout()
plt.show()

The strongest relationships in the dataset are expected to be volume-driven: ZIP codes with more participating households generally have more participants, more rental assistance, and more course attendees. These are descriptive associations and should not be interpreted as causal effects.

## 9. Normalized measures

In [ ]:
normalized = (
    df.groupby("city", as_index=False)
      .agg(
          households=("total_households", "sum"),
          participants=("total_participants", "sum"),
          rental_assistance=("ra_total", "sum"),
          fwc_attendees=("fwc_total", "sum"),
      )
)

normalized["ra_per_household"] = normalized["rental_assistance"] / normalized["households"]
normalized["fwc_attendees_per_100_participants"] = (
    normalized["fwc_attendees"] / normalized["participants"] * 100
)

normalized.sort_values("ra_per_household", ascending=False)

Normalizing by program size changes the question from “where is the most activity?” to “how much activity is occurring relative to the number of participating households or participants?” This helps distinguish scale from intensity.

## 10. Key findings

In [ ]:
largest_city = city.loc[city["rental_assistance"].idxmax()]
largest_zip = df.loc[df["ra_total"].idxmax()]

summary = pd.Series({
    "Metro area with highest total RA": largest_city["city"],
    "Highest-RA ZIP": largest_zip["zip_code"],
    "Highest-RA ZIP metro": largest_zip["city"],
    "Highest ZIP RA": largest_zip["ra_total"],
    "Top-10 ZIP share of total RA": top10_share,
    "August statewide RA MoM change (%)": ra_mom["August"],
    "September statewide RA MoM change (%)": ra_mom["September"],
    "August statewide FWC MoM change (%)": fwc_mom["August"],
    "September statewide FWC MoM change (%)": fwc_mom["September"],
})

summary

### Interpretation

The project shows a clear geographic concentration of program activity. Houston carries the largest overall rental-assistance volume, while a relatively small number of ZIP codes account for a substantial share of total assistance.

The month-to-month analysis adds context that a single cumulative map cannot show on its own. Enrollment, rental assistance, and Financial Wellness attendance follow different temporal patterns, and ZIP-level percentage changes vary widely.

For portfolio purposes, the key analytical lesson is that **scale, change, and intensity are different concepts**:

- total dollars identify high-volume areas,
- MoM percentages identify changing areas,
- per-household and per-participant measures normalize for program size,
- maps reveal the geographic pattern behind those statistics.


## 11. Analytical limitations

This project uses synthetic aggregate ZIP-code-level data. Several limitations are important:

- ZIP-level aggregates cannot support participant-level inference.
- Correlations among program measures do not establish causation.
- Month-over-month percentages can be volatile when the prior-month count is small.
- Cumulative enrollment is conceptually different from monthly RA and monthly FWC attendance.
- Monthly FWC values count unique attendees within each month, while `fwc_total` counts unique attendees across the full July–September period.

The analysis is therefore best interpreted as descriptive program-monitoring and geospatial analytics.
